`requirements/gpu.txt`

In [ ]:
import pandas as pd, gc, warnings
from qiskit_aer.noise import NoiseModel
from qiskit_aer import AerSimulator
from helpers.load_payload import load_payload
from helpers.connected_subsets import connected_subsets
from helpers.quantum_volume import quantum_volume, quantum_volume_optimised
from helpers.compare_circuits import compare_circuits


In [2]:
def noisy_simulator_from_payload(payload):
    '''Creates a noisy AerSimulator from the noise model and 
    coupling map in the payload.'''
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        noise_model = NoiseModel.from_dict(payload["noise_model"])
        
    coupling_map = payload.get("coupling_map")
    return AerSimulator(
        noise_model=noise_model,
        coupling_map=coupling_map,
        basis_gates=noise_model.basis_gates,
        method="automatic", # "statevector",
        device="GPU",
        batched_shots_gpu=True,
    )

In [3]:
def run_qv_over_subsets(backend, subsets, qv_runner):
    '''Runs quantum volume experiments over multiple subsets of qubits 
    and returns a summary of the results.'''

    ideal_backend = AerSimulator(method="statevector", device="GPU")

    summaries = []
    for i, subset in enumerate(subsets, 1):
        print(f"[{i}/{len(subsets)}] running QV on subset {subset}")

        exp_data = qv_runner(
            backend=backend,
            ideal_backend=ideal_backend,
            subset=subset,
            shots=100,
            trials=100,
            seed=42,
        )

        df = exp_data.analysis_results(dataframe=True)
        hop_row = df[df["name"] == "mean_HOP"].iloc[0]

        mean_hop = hop_row["value"].nominal_value
        hop_err  = hop_row["value"].std_dev

        summaries.append((subset, mean_hop, hop_err))

        del exp_data, df
        gc.collect()

    return summaries

In [4]:
qubits = 5 # number of qubits to run QV on
optimised = True # whether to use optimisation

calibration_path = "calibrations/ibm_marrakesh/20260129_101824.json"
payload = load_payload(calibration_path)
coupling_map = payload.get("coupling_map")

subsets = connected_subsets(coupling_map, qubits)

backend = noisy_simulator_from_payload(payload)

In [5]:
expdata = run_qv_over_subsets(
    backend=backend,
    subsets=subsets,
    qv_runner=quantum_volume_optimised if optimised else quantum_volume
)

df = pd.DataFrame(expdata, columns=["subset", "mean_HOP", "hop_error"])
df = df.sort_values("mean_HOP", ascending=False).reset_index(drop=True)

out_csv = f"results/ibm_marrakesh_qv_{qubits}_tuples_{'' if not optimised else 'optimised'}.csv"
df.to_csv(out_csv, index=False)

[1/566] running QV on subset (0, 1, 2, 3, 4)
[2/566] running QV on subset (0, 1, 2, 3, 16)
[3/566] running QV on subset (1, 2, 3, 4, 5)
[4/566] running QV on subset (1, 2, 3, 4, 16)
[5/566] running QV on subset (1, 2, 3, 16, 23)
[6/566] running QV on subset (2, 3, 4, 5, 6)
[7/566] running QV on subset (2, 3, 4, 5, 16)
[8/566] running QV on subset (2, 3, 4, 16, 23)
[9/566] running QV on subset (2, 3, 16, 22, 23)
[10/566] running QV on subset (2, 3, 16, 23, 24)
[11/566] running QV on subset (3, 4, 5, 6, 7)
[12/566] running QV on subset (3, 4, 5, 6, 16)
[13/566] running QV on subset (3, 4, 5, 16, 23)
[14/566] running QV on subset (3, 4, 16, 22, 23)
[15/566] running QV on subset (3, 4, 16, 23, 24)
[16/566] running QV on subset (3, 16, 21, 22, 23)
[17/566] running QV on subset (3, 16, 22, 23, 24)
[18/566] running QV on subset (3, 16, 23, 24, 25)
[19/566] running QV on subset (4, 5, 6, 7, 8)
[20/566] running QV on subset (4, 5, 6, 7, 17)
[21/566] running QV on subset (5, 6, 7, 8, 9)
[22/566]

In [ ]:
ideal_backend = AerSimulator(method="statevector", device="GPU")

all_rows = []

for i, subset in enumerate(subsets, 1):
    print(f"[{i}/{len(subsets)}] subset {subset}")

    df_subset = compare_circuits(
        backend=backend,
        ideal_backend=ideal_backend,
        subset=subset,
        seed=42,
    )

    df_subset["subset"] = [tuple(subset)] * len(df_subset)
    all_rows.append(df_subset)

df_all = pd.concat(all_rows, ignore_index=True)

out_csv = f"results/transpile_compare_{qubits}_tuples.csv"
df_all.drop(columns=["counts"]).to_csv(out_csv, index=False)

7Q: 1591 $\approx$ ??h

8Q: 2800 $\approx$ ??h

9Q: 5054 $\approx$ ??h

10Q: 9286 $\approx$ ??h

11Q: 17343 $\approx$ ??h

12Q: 32617 $\approx$ ??h

13Q: 61477 $\approx$ ??h 